In [ ]:
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ------------------ CONFIG ------------------

TARGET_ROWS = 1500
SAVE_FILE = "amazon_audio.csv"

# Search terms for headphones and speakers
SEARCH_TERMS = [  "headphones", "bluetooth speakers", "neckband", "portable speakers", "earbuds","wired earphones", "tws earbuds", "earphones", 
                  "soundbars", "Noise cancelling headphones"
]

# ------------------ CHROME SETUP ------------------

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

# Disable images for speed
prefs = {"profile.managed_default_content_settings.images": 2}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(options=options)

# ------------------ LOAD OLD DATA ------------------

if os.path.exists(SAVE_FILE):
    df_existing = pd.read_csv(SAVE_FILE)
    existing_asins = set(df_existing['asin'].unique())
    data = df_existing[~df_existing['asin'].duplicated(keep='first')].values.tolist()
    print(f"Resuming from {len(data)} unique products")
else:
    data = []
    existing_asins = set()

print(f"Target: {TARGET_ROWS} unique audio products")
print(f"Using {len(SEARCH_TERMS)} different search terms")

# ------------------ MAIN SCRAPING LOOP ------------------

collected = len(data)
search_term_index = 0
max_pages_without_new = 4
max_pages_per_search = 25

while collected < TARGET_ROWS and search_term_index < len(SEARCH_TERMS):
    
    current_search = SEARCH_TERMS[search_term_index]
    page = 1
    pages_without_new_in_term = 0
    
    print(f"\n--- Search {search_term_index + 1}/{len(SEARCH_TERMS)}: '{current_search}' ---")
    
    while (collected < TARGET_ROWS and 
           page <= max_pages_per_search and 
           pages_without_new_in_term < max_pages_without_new):
        
        if page % 5 == 0:
            print(f"Page {page}: {collected}/{TARGET_ROWS}")

        search_query = current_search.replace(" ", "+")
        url = f"https://www.amazon.in/s?k={search_query}&page={page}"

        success = False
        for attempt in range(3):
            try:
                driver.get(url)

                if "captcha" in driver.page_source.lower():
                    print("CAPTCHA detected. Waiting 60s...")
                    time.sleep(60)
                    continue

                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
                    )
                )

                success = True
                break

            except Exception as e:
                if attempt < 2:
                    time.sleep(5)

        if not success:
            page += 1
            continue

        products = driver.find_elements(
            By.CSS_SELECTOR,
            "div[data-component-type='s-search-result']"
        )
        
        new_products_on_page = 0
        
        for p in products:
            if collected >= TARGET_ROWS:
                break

            try:
                asin = p.get_attribute("data-asin")
                if not asin or len(asin) != 10 or asin in existing_asins:
                    continue

                try:
                    title = p.find_element(By.CSS_SELECTOR, "h2 span").text
                except:
                    title = None

                brand = title.split()[0] if title else None

                try:
                    price_whole = p.find_element(By.CSS_SELECTOR, "span.a-price-whole").text.replace(",", "")
                    try:
                        price_fraction = p.find_element(By.CSS_SELECTOR, "span.a-price-fraction").text
                        price = f"{price_whole}.{price_fraction}"
                    except:
                        price = price_whole
                except:
                    price = None

                try:
                    rating = p.find_element(
                        By.CSS_SELECTOR, "span.a-icon-alt"
                    ).get_attribute("innerHTML").split()[0]
                except:
                    rating = None

                try:
                    review_count = p.find_element(
                        By.CSS_SELECTOR,
                        "span.a-size-base.s-underline-text"
                    ).text
                except:
                    try:
                        review_count = p.find_element(
                            By.CSS_SELECTOR,
                            "span.a-size-mini.puis-normal-weight-text.s-underline-text"
                        ).text.strip("()")
                    except:
                        review_count = None

                try:
                    discount_percent = p.find_element(
                        By.XPATH, ".//span[contains(text(),'% off')]"
                    ).text
                except:
                    discount_percent = None

                link = f"https://www.amazon.in/dp/{asin}"

                data.append([
                    asin, title, brand, price, discount_percent,
                    rating, review_count, "Audio", link, current_search
                ])
                
                existing_asins.add(asin)
                collected += 1
                new_products_on_page += 1

            except Exception as e:
                continue

        if new_products_on_page == 0:
            pages_without_new_in_term += 1
        else:
            pages_without_new_in_term = 0

        if len(data) % 100 == 0:
            columns = ["asin","product_title","brand","price","discount_percent",
                      "rating","review_count","category","product_link","search_term"]
            pd.DataFrame(data, columns=columns).to_csv(SAVE_FILE, index=False)
            print(f"Saved {len(data)} products")

        time.sleep(random.uniform(2, 4))
        page += 1
    
    search_term_index += 1



driver.quit()

columns = ["asin","product_title","brand","price","discount_percent",
          "rating","review_count","category","product_link","search_term"]
df = pd.DataFrame(data, columns=columns)
df.to_csv(SAVE_FILE, index=False)

print(f"\nCompleted! Collected {len(df)} audio products")
if len(df) >= TARGET_ROWS:
    print(f"Reached target of {TARGET_ROWS}")
else:
    print(f"Only collected {len(df)} (target: {TARGET_ROWS})")

print("\nSearch Term Performance:")
term_counts = df['search_term'].value_counts().head(20)
for term, count in term_counts.items():
    print(f"{term}: {count}")

Target: 1500 unique audio products
Using 10 different search terms

--- Search 1/10: 'headphones' ---
Page 5: 67/1500
Page 10: 145/1500
Page 15: 224/1500
Page 20: 298/1500
Saved 300 products
Saved 300 products
Saved 300 products
Saved 300 products
Page 25: 301/1500

--- Search 2/10: 'bluetooth speakers' ---
Page 5: 368/1500
Saved 400 products
Page 10: 450/1500
Page 15: 529/1500
Page 20: 604/1500
Page 25: 607/1500

--- Search 3/10: 'neckband' ---
Page 5: 672/1500
Page 10: 753/1500
Saved 800 products
Page 15: 831/1500
Page 20: 902/1500
Page 25: 907/1500

--- Search 4/10: 'portable speakers' ---
Page 5: 926/1500
Page 10: 949/1500
Page 15: 986/1500
Page 20: 1036/1500

--- Search 5/10: 'earbuds' ---
Page 5: 1103/1500
Page 10: 1175/1500
Page 15: 1255/1500
Page 20: 1322/1500
Page 25: 1326/1500

--- Search 6/10: 'wired earphones' ---
Page 5: 1392/1500
Page 10: 1455/1500
Saved 1500 products

Completed! Collected 1500 audio products
Reached target of 1500

Search Term Performance:
bluetooth spea